In [1]:
!pip install langchain langchain_openai langchain_groq langchain_community langgraph ipykernel python-dotenv
!pip install langgraph-checkpoint-sqlite gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 81.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
ERROR: pip's dependency 

In [2]:
import gdown
url = 'https://drive.google.com/file/d/1f-X_cbCcJG0JrJl2FsCNceuA6POKjnXm/view?usp=drive_link'
output_path = '.env'
gdown.download(url, output_path, quiet=False,fuzzy=True)

Downloading...
From: https://drive.google.com/uc?id=1f-X_cbCcJG0JrJl2FsCNceuA6POKjnXm
To: /content/.env
100%|██████████| 71.0/71.0 [00:00<00:00, 191kB/s]


'.env'

In [3]:
"""
Instructor Solution: LangChain Single Agent with Tools
This script shows how to build a simple agent using LangChain's ReAct-style agent
with three simulated APIs (weather, route, clothing), powered by Groq.
"""

import os, random
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import StructuredTool
from langchain.agents import create_agent

# Load API key
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

# --------------------------
# Define Tools
# --------------------------
# NOTE: We use StructuredTool.from_function() here, not the older Tool.from_function().
# Tool.from_function() always exposes a single generic "tool_input" string field to
# the model, no matter how many parameters the wrapped function actually has. That is
# harmless for weather_api/clothing_api (they only take one argument each), but it
# silently breaks route_api(destination, weather), which needs two -- the model could
# only ever supply one value, causing "route_api() missing 1 required positional
# argument: 'weather'". StructuredTool.from_function() instead builds a proper
# multi-field schema from the function's signature, so the model can supply every
# parameter by name.

def weather_api(location: str) -> str:
    """Simulated Weather API: returns random weather for the given location."""
    conditions = ["sunny", "rainy", "stormy", "cloudy", "hot", "cold", "windy"]
    weather = random.choice(conditions)
    return f"The weather in {location} is {weather}."

def route_api(destination: str, weather: str) -> str:
    """Simulated Route Planner: suggests route based on weather."""
    if "rain" in weather or "storm" in weather:
        return f"Best route to {destination}: take public transport, as weather is {weather}."
    else:
        return f"Best route to {destination}: enjoy a scenic walk, since weather is {weather}."


def clothing_api(weather: str) -> str:
    """Simulated Clothing API: recommends clothing based on weather."""
    if "sunny" in weather or "hot" in weather:
        return "Wear light clothes, sunglasses, and a hat."
    elif "rain" in weather or "storm" in weather:
        return "Wear a raincoat, boots, and carry an umbrella."
    elif "cold" in weather:
        return "Wear a warm jacket, scarf, and gloves."
    else:
        return "Wear comfortable clothes suitable for mild weather."

weather_tool = StructuredTool.from_function(
    func=weather_api,
    name="WeatherAPI",
    description="Get the simulated current weather condition for a specific location. Input: a city or place name. Output: a short weather description."
)

route_tool = StructuredTool.from_function(
    func=route_api,
    name="RouteAPI",
    description="Suggest the best route to a travel destination based on the current weather condition. Inputs: destination name and weather description."
)

clothing_tool = StructuredTool.from_function(
    func=clothing_api,
    name="ClothingAPI",
    description="Recommend suitable clothing based on the current weather condition. Input: weather description (e.g., sunny, rainy, cold). Output: clothing recommendation."
)




# --------------------------
# Build Agent
# --------------------------
# Swap 'llama-3.3-70b-versatile' for any model available on your Groq account
# (e.g. 'llama-3.1-8b-instant', 'openai/gpt-oss-120b', 'moonshotai/kimi-k2-instruct').
chat_model = ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api_key)

system_prompt = SystemMessage(
    content="You are a travel assistant helping users with weather, route planning, and clothing advice."
)

# Use the wrapped Tool objects (with their explicit descriptions) rather than the
# raw functions, so the agent sees the descriptions written for it above.
tools = [weather_tool, route_tool, clothing_tool]

# Create the agent
agent = create_agent(chat_model, tools, system_prompt=system_prompt)


In [4]:
# --------------------------
# Test Queries
# --------------------------
queries = [
    "What is the weather like in Paris today?",
    "I am traveling to Tokyo tomorrow. Can you tell me the weather and what I should wear?",
    "If I go to London today, what’s the weather and what route should I take?",
    "It’s raining in Singapore. How should I travel to the Marina Bay area, and what should I wear?",
    "I’m planning a trip to New York tomorrow. Please tell me the weather, the best route to Central Park, and what clothing to pack."
]

for q in queries:
    print(f"\n--- Query: {q}")
    response = agent.invoke({"messages": [HumanMessage(q)]})
    for msg in response["messages"]:
        print(f"{msg.__class__.__name__}: {msg.content}")


--- Query: What is the weather like in Paris today?
HumanMessage: What is the weather like in Paris today?
AIMessage: 
ToolMessage: The weather in Paris is windy.
AIMessage: 
ToolMessage: Wear comfortable clothes suitable for mild weather.
AIMessage: If you're planning to travel to Paris, I can also help you with the best route. Please let me know your starting location, and I'll be happy to assist you.


ToolMessage: Best route to Paris: enjoy a scenic walk, since weather is windy.
AIMessage: You're all set for a trip to Paris. Enjoy your travels.

--- Query: I am traveling to Tokyo tomorrow. Can you tell me the weather and what I should wear?
HumanMessage: I am traveling to Tokyo tomorrow. Can you tell me the weather and what I should wear?
AIMessage: 
ToolMessage: The weather in Tokyo is stormy.
ToolMessage: Wear light clothes, sunglasses, and a hat.
AIMessage: 
ToolMessage: Wear a raincoat, boots, and carry an umbrella.
AIMessage: I hope you have a safe trip to Tokyo. If you need 